# Phase 3 — Hive Metastore Validation

Proves that Spark sessions share a **persistent SQL catalog** backed by Hive Metastore + PostgreSQL.

## What we validate

| Step | Claim |
|---|---|
| 1 | Spark session uses the Hive catalog (not in-memory) |
| 2 | Databases and tables can be created via SQL |
| 3 | Data written by one session survives a session restart |
| 4 | A second user sees the same catalog without re-creating anything |
| 5 | Window functions and aggregations run correctly on Hive-managed tables |

**Kernel**: select **PySpark** (SparkMagic) — not Python 3.

---

## Architecture reminder

```
SparkMagic kernel  →  Livy (driver)  →  spark-worker (executors)
                                              |
                              writes Parquet to /opt/hive/data/warehouse
                              (shared Docker volume: hive-warehouse)
                                              |
                       hive-metastore  ←─────┘  (records table location in postgres)
```

## Step 0 — Verify Livy session

In [ ]:
%%info

## Step 1 — Confirm Hive catalog is active

**Pass condition**: prints `hive` and the HMS thrift URL.

In [ ]:
catalog_impl = spark.conf.get("spark.sql.catalogImplementation")
metastore_uri = spark.conf.get("spark.hadoop.hive.metastore.uris")

print(f"Catalog implementation : {catalog_impl}")
print(f"Metastore URI          : {metastore_uri}")

assert catalog_impl == "hive", f"Expected 'hive', got '{catalog_impl}' -- check config.json session_configs"
print("PASS: Hive catalog active")

## Step 2 — Create the risk_dw database and trades table

Tables are stored as **Parquet** in the shared `hive-warehouse` Docker volume.

In [ ]:
%%sql
CREATE DATABASE IF NOT EXISTS risk_dw
COMMENT 'Risk data warehouse -- Phase 3 PoC'

In [ ]:
%%sql
SHOW DATABASES

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS risk_dw.trades (
    trade_id    STRING    COMMENT 'Unique trade identifier',
    trader      STRING    COMMENT 'Trader login name',
    instrument  STRING    COMMENT 'Ticker symbol',
    quantity    INT       COMMENT 'Number of shares',
    price       DOUBLE    COMMENT 'Execution price (USD)',
    trade_date  DATE      COMMENT 'Settlement date'
)
STORED AS PARQUET
TBLPROPERTIES ('created_by' = 'phase3-validation')

In [ ]:
%%sql
DESCRIBE EXTENDED risk_dw.trades

## Step 3 — Load synthetic trade data

Generates 10 000 realistic trades (4 traders × 5 instruments × 12 months).  
Written to the Hive warehouse as Parquet via `saveAsTable`.

In [ ]:
from pyspark.sql import Row
from datetime import date
import random

random.seed(42)

TRADERS     = ["alice", "bob", "carol", "dave"]
INSTRUMENTS = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]
N           = 10_000

rows = [
    Row(
        trade_id   = f"T{i:06d}",
        trader     = random.choice(TRADERS),
        instrument = random.choice(INSTRUMENTS),
        quantity   = random.randint(100, 10_000),
        price      = round(random.uniform(100.0, 500.0), 2),
        trade_date = date(2025, random.randint(1, 12), random.randint(1, 28)),
    )
    for i in range(N)
]

df = spark.createDataFrame(rows)
df.write.mode("overwrite").saveAsTable("risk_dw.trades")

count = spark.table("risk_dw.trades").count()
print(f"Loaded {count:,} rows into risk_dw.trades")
assert count == N

## Step 4 — Analytical queries

Real bank use cases: notional by trader, instrument concentration, monthly flow.

In [ ]:
%%sql
-- Notional traded per trader (sorted descending)
SELECT
    trader,
    COUNT(*)                              AS trades,
    SUM(quantity)                         AS total_shares,
    ROUND(SUM(quantity * price), 2)       AS notional_usd
FROM risk_dw.trades
GROUP BY trader
ORDER BY notional_usd DESC

In [ ]:
%%sql
-- Top 2 instruments by trade count per trader (window function)
SELECT trader, instrument, trades
FROM (
    SELECT
        trader,
        instrument,
        COUNT(*) AS trades,
        RANK() OVER (PARTITION BY trader ORDER BY COUNT(*) DESC) AS rnk
    FROM risk_dw.trades
    GROUP BY trader, instrument
) t
WHERE rnk <= 2
ORDER BY trader, rnk

In [ ]:
%%sql
-- Monthly notional flow (useful for limit monitoring)
SELECT
    MONTH(trade_date)                   AS month,
    instrument,
    ROUND(SUM(quantity * price), 0)     AS notional_usd
FROM risk_dw.trades
GROUP BY MONTH(trade_date), instrument
ORDER BY month, notional_usd DESC

## Step 5 — Persistence test (session restart)

**Restart the Livy session now** using the SparkMagic widget (`%manage_spark` or the toolbar button),  
then run the cells below.

**Pass condition**: `risk_dw.trades` still exists with 10 000 rows — without re-running Step 3.

In [ ]:
# Run this AFTER restarting the Livy session
count = spark.table("risk_dw.trades").count()
print(f"Rows after session restart: {count:,}")
assert count == 10_000, f"Expected 10000, got {count} -- HMS or warehouse volume not working"
print("PASS: table persisted across session restart")

In [ ]:
%%sql
SHOW TABLES IN risk_dw

## Step 6 — Cross-user catalog visibility

Log in as **bob** in a second browser tab, open a new notebook with the PySpark kernel,  
and run this cell — `risk_dw.trades` must be visible without any setup.

This is the core value of a shared metastore: **one user creates, all users discover**.

In [ ]:
%%sql
-- Run this as bob (or any other user) in a fresh session
-- No CREATE DATABASE, no CREATE TABLE needed
SELECT trader, COUNT(*) AS trades
FROM risk_dw.trades
GROUP BY trader
ORDER BY trades DESC

## Step 7 — Inspect the warehouse on disk

This `%%local` cell runs in the singleuser container and calls the Livy REST API  
to show all active sessions — both alice and bob should appear simultaneously.

In [ ]:
%%local
import urllib.request, json

with urllib.request.urlopen("http://livy:8998/sessions") as r:
    data = json.loads(r.read())

print(f"Active Livy sessions: {data['total']}")
for s in data["sessions"]:
    print(f"  id={s['id']}  state={s['state']}  kind={s['kind']}  appId={s.get('appId','--')}")

## Summary

| Check | How to verify | Pass condition |
|---|---|---|
| Hive catalog active | Step 1 | `spark.sql.catalogImplementation = hive` |
| Database + table created | Step 2 | `SHOW DATABASES` lists `risk_dw` |
| Data written to warehouse | Step 3 | `count == 10 000` |
| SQL analytics work | Step 4 | Aggregation and window function return rows |
| Persistence across restarts | Step 5 | After session kill, `count` still 10 000 |
| Cross-user catalog | Step 6 | bob queries table alice created |

### Production equivalence

| This stack | Production |
|---|---|
| Postgres in Docker | Managed RDS / Cloud SQL |
| `/opt/hive/data/warehouse` (Docker volume) | `hdfs:///user/hive/warehouse` |
| No auth on HMS | SASL + Kerberos (Phase 5) |
| Single HMS instance | HMS with ZooKeeper HA |